# Creating a basic chat experience with context variables

In this example, we show how you can build a simple chat bot by sending and updating context with your requests. 

We introduce the Context Variables object which in this demo functions similarly as a key-value store that you can use when running the kernel.

The context is local (i.e. in your computer's RAM) and not persisted anywhere beyond the life of this Jupyter session.

In future examples, we will show how to persist the context on disk so that you can bring it into your applications.  

In this chat scenario, as the user talks back and forth with the bot, the context gets populated with the history of the conversation. During each new run of the kernel, the context can provide the AI with its variables' content. 

In [1]:
#r "nuget: Microsoft.SemanticKernel, 1.0.0-beta1"
#!import config/Settings.cs

using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.SemanticFunctions;
using Microsoft.SemanticKernel.Orchestration;
using Microsoft.SemanticKernel.Connectors.AI.OpenAI;

var builder = new KernelBuilder();

// Configure AI backend used by the kernel
var (useAzureOpenAI, model, azureEndpoint, apiKey, orgId) = Settings.LoadFromFile();
if (useAzureOpenAI)
    builder.WithAzureChatCompletionService(model, azureEndpoint, apiKey);
else
    builder.WithOpenAIChatCompletionService(model, apiKey, orgId);

IKernel kernel = builder.Build();

Installed Packages Microsoft.SemanticKernel, 1.0.0-beta1

Let's define a prompt outlining a dialogue chat bot.

In [3]:
const string skPrompt = @"
ChatBot can have a conversation with you about any topic.
It can give explicit instructions or say 'I don't know' if it does not have an answer.

{{$history}}
User: {{$userInput}}
ChatBot:";

var aiRequestSettings = new OpenAIRequestSettings 
{
    MaxTokens = 2000,
    Temperature = 0.7,
    TopP = 0.5
};

var promptConfig = new PromptTemplateConfig();
promptConfig.ModelSettings.Add(aiRequestSettings);

Register your semantic function

In [4]:
var promptTemplate = new PromptTemplate(skPrompt, promptConfig, kernel);
var functionConfig = new SemanticFunctionConfig(promptConfig, promptTemplate);
var chatFunction = kernel.RegisterSemanticFunction("ChatBot", "Chat", functionConfig);

Initialize your context

In [5]:
var context = kernel.CreateNewContext();

var history = "";
context.Variables["history"] = history;

Chat with the Bot

In [6]:
var userInput = "Hi, I'm looking for book suggestions";
context.Variables["userInput"] = userInput;

var bot_answer = await chatFunction.InvokeAsync(context);

Update the history with the output and set this as the new input value for the next request

In [7]:
history += $"\nUser: {userInput}\nMelody: {bot_answer.GetValue<string>()}\n";
context.Variables.Update(history);

Console.WriteLine(context);


User: Hi, I'm looking for book suggestions
Melody: Of course! I'd be happy to help you with book suggestions. What genre or type of books are you interested in?



Keep Chatting!

In [8]:
Func<string, Task> Chat = async (string input) => {
    // Save new message in the context variables
    context.Variables["userInput"] = input;

    // Process the user message and get an answer
    var answer = await chatFunction.InvokeAsync(context);

    // Append the new interaction to the chat history
    history += $"\nUser: {input}\nMelody: {answer.GetValue<string>()}\n"; 
    context.Variables["history"] = history;
    
    // Show the response
    Console.WriteLine(context);
};

In [9]:
await Chat("I would like a non-fiction book suggestion about Greece history. Please only list one book.");

I would recommend "The Peloponnesian War" by Thucydides. It is a classic work of ancient Greek history that provides a detailed account of the war between Athens and Sparta.


In [10]:
await Chat("that sounds interesting, what are some of the topics I will learn about?");

In "The Peloponnesian War," you will learn about the causes and events of the war, including the political and military strategies of Athens and Sparta. You will also gain insights into the social and cultural aspects of ancient Greece, such as the role of democracy, the power dynamics between city-states, and the impact of the war on Greek society. Thucydides' work is known for its analytical approach and its exploration of human nature and the nature of power.


In [11]:
await Chat("Which topic from the ones you listed do you think most people find interesting?");

I think many people find the political and military strategies of Athens and Sparta during the Peloponnesian War to be particularly interesting. The war was a major conflict between two powerful city-states, and understanding the strategies employed by each side can provide valuable insights into ancient Greek warfare and politics. Additionally, the impact of the war on Greek society and the exploration of human nature and the nature of power are also topics that tend to captivate readers.


In [12]:
await Chat("could you list some more books I could read about the topic(s) you mentioned?");

Certainly! Here are a few more books you could consider:

1. "The Spartans: The World of the Warrior-Heroes of Ancient Greece" by Paul Cartledge - This book provides an in-depth exploration of the Spartan society and its military prowess, shedding light on the role of Sparta in the Peloponnesian War.

2. "The Athenian Democracy in the Age of Demosthenes" by Mogens Herman Hansen - This book delves into the political system of Athens during the time of the Peloponnesian War, examining the workings of Athenian democracy and the role of influential figures like Demosthenes.

3. "The Greek World 479-323 BC" by Simon Hornblower - This book offers a comprehensive overview of Greek history during the period that includes the Peloponnesian War, covering topics such as politics, warfare, and culture.

4. "The Fall of the Athenian Empire" by Donald Kagan - This book focuses on the decline and fall of Athens after the Peloponnesian War, exploring the consequences of the war and its impact on Athen

After chatting for a while, we have built a growing history, which we are attaching to each prompt and which contains the full conversation. Let's take a look!

In [ ]:
Console.WriteLine(context.Variables["history"]);